Subject: ST 554 - Final Project

Name: Franklin Zhou

Date: 4/19/2026

# Fitting Your Model (50 pts)

**Create a Jupyter notebook for the modeling fitting part and the Streaming part below.**

- The file `power_ml_data.csv` is available at the URL: https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv
- You should read this data into a standard pandas data frame using the `pd.read_csv()` function.
- Convert this to a spark data frame
- We are going to treat the `Power_Zone_3` variable as our response variable.
- We can use all of the other variables as predictors. (Imagine we know that the `Power_Zone_3` reading is going to go offline in the future and we need to be able to predict that value appropriately.)

In [13]:
# Load packages and initiate spark session
import pandas as pd
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

In [14]:
# Read data
ml_data = pd.read_csv("https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv")
df = spark.createDataFrame(ml_data) # convert to spark sql data frame
df.show(5)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
+-----------+--------+----------+-------

We want to fit an elastic net model using CV (no training/test split, just using CV on the data we’ve read in) with the steps below. 

The transformations below should each use an `MLlib` function that can be put into a pipeline

- The Hour column is likely not stored as a `DoubleType`. If it is not, use an SQL transformer to cast the variable as a `DoubleType`


In [15]:
# Load packages
from pyspark.ml.feature import SQLTransformer, VectorAssembler, Binarizer, OneHotEncoder, StringIndexer, PCA
from pyspark.ml import Pipeline


In [16]:
# Check the schema
df.schema

StructType([StructField('Temperature', DoubleType(), True), StructField('Humidity', DoubleType(), True), StructField('Wind_Speed', DoubleType(), True), StructField('General_Diffuse_Flows', DoubleType(), True), StructField('Diffuse_Flows', DoubleType(), True), StructField('Power_Zone_1', DoubleType(), True), StructField('Power_Zone_2', DoubleType(), True), StructField('Power_Zone_3', DoubleType(), True), StructField('Month', LongType(), True), StructField('Hour', LongType(), True)])

In [17]:
# Cast Hour to Double Type and rename Power_Zone_3 as label
cast_sql = SQLTransformer(
    statement = """
        SELECT *, CAST(Hour AS DOUBLE) AS Hour_Double
        FROM __THIS__
    """
)

- Binarize the `Hour` column based on the column being less than 6.5 or not (night vs day essentially)

In [18]:
# Binarize Hour_Double: 1 if Hour < 6.5 (night), 0 otherwise (day).
binarizer = Binarizer(
    inputCol = "Hour_Double",
    outputCol = "Hour_Bin",
    threshold = 6.5
)

- One-hot encode the Month column

In [19]:
# Cast Month to string first via a SQLTransformer, then index and encode.
cast_month_sql = SQLTransformer(
    statement = "SELECT *, CAST(Month AS STRING) AS Month_str FROM __THIS__"
)

# StringIndexer maps each month a numeric index
month_indexer = StringIndexer(
    inputCol = "Month_str",
    outputCol = "Month_idx"
)

# OneHotEncoder converts numeric index to binary vector
month_encoder = OneHotEncoder(
    inputCol = "Month_idx",
    outputCol = "Month_vector"
)

- Run a PCA fit on the `Temperature`, `Humidity`, `Wind_Speed`, `General_Diffuse_Flows`, and `Diffuse_Flows` columns.

     - To do this, I first used a `VectorAssembler()` call to place these variables in a column together for use with the `PCA()` estimator.
    
     - Once fitted, then you’ll have a PCA transformer we’ll use in our pipeline.
    
    - We’ll use two PCs in our transformation.

In [20]:
pca_assembler = VectorAssembler(
    inputCols = ["Temperature", "Humidity", "Wind_Speed", "General_Diffuse_Flows", "Diffuse_Flows"], 
    outputCol = "PCA_input"
)

pca = PCA(
    k = 2, 
    inputCol = "PCA_input", 
    outputCol = "PCA_features"
)

- Rename your response variable as `label`

In [21]:
# Rename Power_Zone_3 to label
label_sql = SQLTransformer(
    statement = "SELECT *, Power_Zone_3 AS label FROM __THIS__"
)

- Use VectorAssembler() to put your predictors into a features. Use the
     - two fitted PCA features
     - binary `Hour` variable
     - `Power_Zone_1`
     - `Power_Zone_2`
     - `Month` indicator variables


In [22]:
# Combine all predictors into the 'features' vector column.
features_assembler = VectorAssembler(
    inputCols = ["PCA_features", "Hour_Bin", "Power_Zone_1", "Power_Zone_2", "Month_vector"],
    outputCol = "features"
)

- This ends the pipeline of transformations!

- Now you’ll then use the `CrossValidator()` function and the LinearRegression() function to fit an elastic net model.

    - You should do the following grid for the regParam and elasticNetParam: All combinations of
    
        - regParam: 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1
        
        - elasticNetParam: 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1
        
- Now fit the model using 5-fold CV with `rmse` as your criterion!
        

In [23]:
# Load packages
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator

In [24]:
# Setup LinearRegression instance
lr = LinearRegression()

# Setup parameters grid
paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .addGrid(lr.elasticNetParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .build()

# Setup pipeline
transformation_pipeline = Pipeline(stages=[cast_sql, binarizer, cast_month_sql, month_indexer, month_encoder, pca_assembler, pca, label_sql, features_assembler, lr])

# Create cross validation instance
crossval_lr = CrossValidator(estimator = transformation_pipeline,
                          estimatorParamMaps = paramGrid,
                          evaluator = RegressionEvaluator(metricName = 'rmse'),
                          numFolds = 5)

In [25]:
# Fit the cv model
cv_model = crossval_lr.fit(df)

26/04/19 20:48:57 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/04/19 20:48:57 WARN Instrumentation: [9a5f6289] regParam is zero, which might cause numerical instability and overfitting.
26/04/19 20:49:00 WARN Instrumentation: [e73fb814] regParam is zero, which might cause numerical instability and overfitting.
26/04/19 20:49:02 WARN Instrumentation: [ae0e835c] regParam is zero, which might cause numerical instability and overfitting.
26/04/19 20:49:03 WARN Instrumentation: [6a218cae] regParam is zero, which might cause numerical instability and overfitting.
26/04/19 20:49:05 WARN Instrumentation: [9bb6b14b] regParam is zero, which might cause numerical instability and overfitting.
26/04/19 20:49:06 WARN Instrumentation: [a8e9796c] regParam is zero, which might cause numerical instability and overfitting.
26/04/19 20:49:08 WARN Instrumentation: [25a78f41] regP

- Report the optimal values chosen for the tuning parameters

- Report the CV error

In [26]:
# Create a list contains RMSE value associate with parameters value
my_list = []
for i in range(len(paramGrid)):
    my_list.append([cv_model.avgMetrics[i], paramGrid[i].values()])

import numpy as np
# Convert to numpy array 
arrange = np.array(my_list)
# Sort by RMSE value
my_list_sorted = arrange[arrange[:, 0].argsort()]

# Print top 5 rows
print(my_list_sorted[:5])

[[2147.6805955040063 dict_values([0.25, 0.9])]
 [2147.6806262880464 dict_values([0.25, 0.95])]
 [2147.680652374175 dict_values([0.75, 0.25])]
 [2147.6806816732246 dict_values([0.25, 0.99])]
 [2147.680687594257 dict_values([0.25, 1.0])]]


From the output we find that the minumum RMSE is 2147.6805955040063 while the best `regParam` value is 0.25 and the best `elasticNetParam` value is 0.9.

In [27]:
# Another way to extract the value
best_lr = cv_model.bestModel.stages[-1]
# Retrieve optimal tuning parameters
print("Best regParam value is:", best_lr._java_obj.getRegParam())
print("Best elasticNetParam value is:", best_lr._java_obj.getElasticNetParam())
# CV RMSE 
print("CV RMSE:", min(cv_model.avgMetrics))

Best regParam value is: 0.25
Best elasticNetParam value is: 0.9
CV RMSE: 2147.6805955040063


- Report the training set RMSE (as done in the notes) by using your fitted model as a transformer and evaluating on the entire training set

In [28]:
# Now cv_model is a transformer with "best model" as default.
train_predictions = cv_model.transform(df)
training_rmse = RegressionEvaluator(metricName = "rmse").evaluate(train_predictions)
print("The training RMSE value is:", training_rmse)

The training RMSE value is: 2147.0975198849655


- Take the outputted transformations from the model (the predictions) and create a `residual` column (`label` - `prediction`). The `.withColumn()` method is handy here. Print out a data frame with these `residual`s, the `label` column, and the `prediction`s

In [29]:
from pyspark.sql.functions import col
# create column residual = label - prediction
res_df = train_predictions.withColumn("residual", col("label") - col("prediction")) \
             .select("label","prediction","residual")

res_df.show(10)

+-----------+------------------+------------------+
|      label|        prediction|          residual|
+-----------+------------------+------------------+
|20240.96386|20879.059021624707|-638.0951616247075|
|20131.08434|18658.584292356452| 1472.500047643549|
|19668.43373|18203.134666224345|1465.2990637756557|
|18899.27711| 17589.12367656089|1310.1534334391108|
|18442.40964| 16995.81374405468|  1446.59589594532|
|18130.12048|16516.251987204687|1613.8684927953145|
|17945.06024|16091.866359951036|1853.1938800489625|
|17459.27711|15721.348901258327|1737.9282087416723|
|17025.54217|15269.749670036235|1755.7924999637653|
|16794.21687|  14937.0805978629|1857.1362721371006|
+-----------+------------------+------------------+
only showing top 10 rows


## Streaming Part (40 pts)

There is another file available at: https://www4.stat.ncsu.edu/~online/datasets/power_streaming_data.csv

Download this file and store it where your `.py` file you’ll create can find it. We’ll be randomly sampling rows from this to output to `.csv` files that you’ll be reading in.

### Reading a Stream

- We’re going to read in a stream in the form of `.csv` files. Create a folder where you will be sending your `.csv` files.
- Setup the schema for the stream (you can use the schema from the original data as we did in hw 10)
- Set up the `readStream`. Be sure to add `header = True` as you’ll likely be outputting files with a header and we don’t need to read that in.

In [36]:
data_schema = df.schema

In [37]:
# read stream data from Final_Project/stream_folder folder
stream_df = spark.readStream.option("header", True).schema(data_schema).csv("stream_folder")

### Transform/Aggregation Step

- Now, we’ll do two separate things on the stream and join them together:
    - With your stream, use your model transformer to obtain predictions from the incoming data. On the resulting predictions also create a `residual` column as noted in the previous section (return only the `label`, `prediction` and `residual` columns from this part)
    - We can use our stream more than once! With another transformation on the (original) stream, modify the response variable to be called `label`.
    - Now join your above transform with this stream based on the `label` variable which should be common to both!
    - Note 1: This is a little silly, but I want you to join two transformations of the stream and I don’t want things to get too crazy
    - Note 2: Each data frame is created from the same stream of data! You don’t need two streams, you can use the same stream and just do two separate transformations on it, combining it with a `.join()` method from one of the SQL style data frames you are dealing with (as we discussed in the notes)

In [38]:
# use model transformer to obtain predictions from the incoming data
stream_predictions = cv_model.transform(stream_df)

# Add residual column and keep only the three required columns
stream_with_residuals = stream_predictions.withColumn("residual", col("label") - col("prediction")).select("label", "prediction", "residual")

In [39]:
# Transformation 2: Rename the response variable
stream_label = stream_df.withColumnRenamed("Power_Zone_3", "label")

# Inner join the two stream transformations by label
joined_stream = stream_with_residuals.join(stream_label, on = "label", how = "inner")

### Writing Step

- Now write your stream to the console using the append output mode.

- Start the query!

In [40]:
# Start the query
stream_query = (
    joined_stream
    .writeStream
    .outputMode("append")
    .format("console")
    .start()
)

26/04/19 21:30:23 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-cac7f5dc-721e-4e61-adbd-5feb5e790930. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/19 21:30:23 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|16429.87952|11573.964715914044| 4855.9148040859545|      21.09|   62.51|     0.071|                445.0|        389.3| 30203.07692| 24396.69421|   11|  14|
|14036.67692|16136.386432409963|-2099.7095124099633|      18.06|    72.4|     4.918|                0.055|        0.137| 24909.13907| 15387.94179|    6|   5|
|25231.80723|25751.613961697803| -519.8067316978049|      15.42|    72.1|     0.073|                4.604|       

-------------------------------------------
Batch: 1
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|9882.352941|11541.686762526711|-1659.3338215267122|      16.56|   58.08|     0.087|                209.9|         93.1| 31063.11787| 25259.28199|   12|  13|
|26124.85356| 30520.43571247156| -4395.582152471561|      27.23|   44.82|     4.904|                720.0|        45.57| 41002.52492| 29335.44304|    7|  10|
|7266.266507| 6042.253878389567| 1224.0126286104332|      15.38|   67.15|     0.071|                 24.1|       

-------------------------------------------
Batch: 2
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|9594.216867|10698.643639091832|-1104.4267720918324|      16.91|   64.42|     4.916|                0.044|          0.1| 23433.84615| 18639.66942|   11|   6|
|22052.91536|22340.422952928355|-287.50759292835573|      24.91|   58.85|     4.914|                479.7|        27.77| 34246.74806|  24093.7698|    8|   9|
|11811.79331|  13409.4649991548|-1597.6716891548003|       22.7|   69.85|      4.92|                266.2|       

-------------------------------------------
Batch: 3
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|11953.73494|11964.905142297423|-11.170202297422293|       19.2|    74.9|     0.069|                0.069|          0.1| 25415.38462| 19249.58678|   11|   0|
|15028.96552|  18824.5684317122|   -3795.6029117122|      21.93|   69.16|     4.914|                2.259|        1.774| 25181.62042| 15711.51003|    8|   6|
|18083.85542|14161.140576727641|  3922.714843272359|      22.05|   52.77|     0.072|                462.5|       

...

-------------------------------------------
Batch: 13
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|16956.14458|18710.491361852208|-1754.3467818522076|      15.82|   65.32|     0.075|                180.7|        173.7| 33940.25316| 21257.14286|    1|  15|
|25377.74059| 25072.08350647366|   305.657083526341|      26.19|   42.15|     4.917|                448.2|        363.2| 33647.84053| 23020.25316|    7|   8|
|19729.65517| 25989.63607200335|  -6259.98090200335|       27.2|   47.22|     4.933|                135.5|      

-------------------------------------------
Batch: 14
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|14767.59799|13182.790294680457| 1584.8076953195432|        8.2|   54.42|     0.092|                0.051|        0.119| 22252.88136|  12955.6231|    2|   2|
|10457.87234|11322.332148484762| -864.4598084847621|      18.51|    90.3|     0.066|                31.55|        26.35| 30262.05689| 20236.92946|   10|   8|
|17936.05016| 21211.55274682326|-3275.5025868232624|      24.12|    82.0|     4.921|                225.0|      

-------------------------------------------
Batch: 15
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|          residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|16886.74699| 16618.97979247828|267.76719752171994|        8.3|    78.6|     0.087|                0.037|        0.152| 26782.78481| 17482.06687|    1|   0|
|16828.11715|22513.399106859735|-5685.281956859737|      21.07|    79.4|     4.905|                0.293|        0.308| 24430.56478| 16017.72152|    7|   5|
|9813.975904| 9677.219961602084|136.75594239791644|       7.26|    80.4|     4.918|                0.073|        0.12

-------------------------------------------
Batch: 16
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|16048.01921|16879.030810644956| -831.0116006449553|       14.0|   65.99|     4.955|                 0.07|        0.133| 37742.96578| 30535.74716|   12|  19|
|26318.76923|27574.116198392825|-1255.3469683928233|      20.45|    82.6|     0.071|                0.062|        0.115| 45062.78146| 24578.79418|    6|  22|
|26085.51724|26040.648213139855|  44.86902686014582|       26.8|    86.1|     4.909|                244.3|      

-------------------------------------------
Batch: 17
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|10224.43769|12845.471935936006|-2621.0342459360054|      20.31|    77.9|     4.915|                110.5|         99.7| 33123.15098| 21850.20747|   10|   9|
|10363.37349|13799.380479533822| -3436.006989533822|      19.04|   60.73|     0.074|                208.3|        54.61| 30996.92308|  26910.7438|   11|  16|
|10402.73367|10261.443016940015| 141.29065305998483|      11.63|   65.75|     0.088|                 9.65|      

In [ ]:
# Stop the query
stream_query.stop()

# Reference

https://share.google/aimode/nADjDIlL2cBxKdAkP
